# 🧠 CalRetail — Retail Media Network Planner
## Goal
Match brand campaigns to targeted consumer segments and calculate ROAS using real historical
campaign performance.

## Algorithmic Explanation
**Funnel Statistics ROAS Forecaster**
1. Query customers dataset matching real segment or preferred-category profiles (previously
   matched against segment values like "VIP"/"Loyal"/"Budget"/"Churned" that don't exist anywhere
   in this dataset's real segment list, so those branches always fell through).
2. Forecast clicks/conversions using each channel's REAL historical CTR/CVR from
   `marketing_campaigns.csv` (previously fixed guesses: Email=2.5%, SMS=3.8%, Display=1.2%).
3. Estimate revenue using the matched segment's real average order value from the engineered RFM
   feature table (previously a flat ₹1200 assumed for every segment/campaign).



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data' / 'processed'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
cust = pd.read_csv(processed_dir / 'customers.csv')
camps = pd.read_csv(processed_dir / 'marketing_campaigns.csv')
feat_cust = pd.read_csv(processed_dir / 'feature_customers.csv')

# Real channel-level CTR/CVR benchmarks measured from actual campaign
# performance history — replaces fixed guesses (Email=2.5%, SMS=3.8%, Display=1.2%).
channel_stats = camps.groupby('channel').agg(avg_ctr=('ctr', 'mean'), avg_cvr=('cvr', 'mean'))
GLOBAL_CTR = float(camps['ctr'].mean())
GLOBAL_CVR = float(camps['cvr'].mean())

# Real average order value per segment from the engineered RFM feature table
# — replaces a flat Rs.1200 guess used for every segment/campaign.
segment_aov = feat_cust.groupby('segment')['avg_order_val'].mean()
GLOBAL_AOV = float(feat_cust['avg_order_val'].mean())

print(f"Customer metrics available. Segment sizes: {cust['segment'].value_counts().to_dict()}")
print(f"Real channel CTR/CVR benchmarks:\n{channel_stats}")

In [ ]:
def generate_media_budget_plan(target_segment, budget, channel):
    global cust
    df_cust = cust.copy()

    seg_low = str(target_segment).lower()
    real_segments_low = df_cust['segment'].str.lower().unique()
    real_categories_low = df_cust['preferred_category'].str.lower().unique()

    # Match the requested target against REAL segment or preferred-category
    # values (previously checked against "VIP"/"Loyal"/"Budget"/"Churned",
    # none of which exist anywhere in this dataset's real segment list, so
    # those branches silently always fell through to the generic sample).
    if seg_low in real_segments_low:
        matched_custs = df_cust[df_cust['segment'].str.lower() == seg_low]
    elif seg_low in real_categories_low:
        matched_custs = df_cust[df_cust['preferred_category'].str.lower() == seg_low]
    else:
        matched_custs = df_cust.sample(n=min(len(df_cust), 1500), random_state=42)

    audience_size = max(len(matched_custs), 1)

    # Real channel CTR/CVR from actual campaign history (falls back to the
    # global average for a channel with no historical campaigns yet).
    if channel in channel_stats.index:
        ctr = float(channel_stats.loc[channel, 'avg_ctr']) / 100.0
        cvr = float(channel_stats.loc[channel, 'avg_cvr']) / 100.0
    else:
        ctr, cvr = GLOBAL_CTR / 100.0, GLOBAL_CVR / 100.0

    # Loyalty-tier mix boosts engagement — a real column, replacing a "VIP"/
    # "Loyal" segment check that never matched any real segment value.
    loyalty_weight = {'Bronze': 0.0, 'Silver': 0.2, 'Gold': 0.5, 'Platinum': 1.0}
    loyalty_boost = float(matched_custs['loyalty_tier'].map(loyalty_weight).fillna(0.2).mean())
    ctr = float(np.clip(ctr * (1.0 + loyalty_boost * 0.3), 0.001, 0.5))
    cvr = float(np.clip(cvr * (1.0 + loyalty_boost * 0.5), 0.001, 0.6))

    impressions = audience_size * 10
    clicks = impressions * ctr
    conversions = clicks * cvr

    # Real average order value for the matched segment, from the engineered
    # RFM feature table — replaces a flat Rs.1200 guess for every campaign.
    seg_match = matched_custs['segment'].mode().iloc[0] if not matched_custs.empty else None
    mean_spend = float(segment_aov.get(seg_match, GLOBAL_AOV)) if seg_match else GLOBAL_AOV

    revenue = conversions * mean_spend
    roas = revenue / budget if budget > 0 else 0.0

    return {
        "target_segment": target_segment,
        "channel": channel,
        "budget": budget,
        "matched_audience_size": int(audience_size),
        "estimated_impressions": int(impressions),
        "estimated_clicks": int(clicks),
        "estimated_conversions": int(conversions),
        "predicted_ctr": round(float(ctr * 100), 2),
        "predicted_cvr": round(float(cvr * 100), 2),
        "predicted_roas": round(float(roas), 2),
        "recommended_placement": "Premium Display Feed placement" if channel == "Display" else "Direct Segment Push Notification",
        "est_audience_reach": int(audience_size),
        "est_impressions": int(impressions),
        "est_clicks": int(clicks),
        "est_revenue": round(float(revenue), 2)
    }

backend_res = generate_media_budget_plan("Frequent Buyer", 10000, "SMS")
print("Campaign budget layout payload:\n", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL MEDIA NETWORK PLANNER ===")
print(f"Optimizing Campaign for: {backend_res['target_segment']} (SMS channel)")
print(f"Ad Budget: ₹{backend_res['budget']} | Expected Revenue returns: ₹{backend_res['est_revenue']}")
print(f"Projected ROAS score: {backend_res['predicted_roas']}x (Reach: {backend_res['est_audience_reach']} shoppers)")
